# Base Notebook – Late Delivery Risk Prediction (Tuần 1)
Phạm vi: **Split → Preprocessing → Evaluation Pipeline**.

Nguyên tắc cố định:
- Test set chỉ dùng 1 lần ở bước đánh giá cuối; mọi chọn model/hyperparameter/ngưỡng làm bằng CV/OOF trên Train.
- Positive class = `Late_Delivery = 1`.
- Imputer/Encoder/Scaler nằm trong Pipeline, chỉ fit trên train của từng fold.
- Cùng 1 danh sách ID train/test cho cả FS1/FS2/FS3.

In [1]:
import sys, platform, json, os
import numpy as np, pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix)

DATA_PATH = 'Train.csv'
SEEDS = [42, 123, 2024, 2025, 2026]   # 42 = main
MAIN_SEED = 42
TARGET_RAW, TARGET = 'Reached.on.Time_Y.N', 'Late_Delivery'
os.makedirs('splits', exist_ok=True); os.makedirs('results', exist_ok=True)
print(sys.version.split()[0], 'sklearn', sklearn.__version__, 'pandas', pd.__version__, 'numpy', np.__version__)

3.12.10 sklearn 1.9.0 pandas 3.1.0 numpy 2.5.1


## 1. Load & chuẩn hóa target
Ghi chú: file gốc `Reached.on.Time_Y.N` = 1 là **trễ**.

In [2]:
df = pd.read_csv(DATA_PATH).rename(columns={TARGET_RAW: TARGET})
assert df['ID'].is_unique and df.isna().sum().sum() == 0
print(df.shape); print(df[TARGET].value_counts().rename('n').to_frame().assign(rate=lambda x: x.n/len(df)))

(10999, 12)
                  n      rate
Late_Delivery                
1              6563  0.596691
0              4436  0.403309


## 2. Feature sets (FS1/FS2/FS3)

In [3]:
CAT = ['Warehouse_block','Mode_of_Shipment','Product_importance','Gender']
NUM = ['Customer_care_calls','Customer_rating','Cost_of_the_Product','Prior_purchases','Discount_offered','Weight_in_gms']
FS = {
 'FS1': ['Warehouse_block','Mode_of_Shipment','Customer_care_calls','Customer_rating','Cost_of_the_Product','Prior_purchases','Product_importance','Gender','Discount_offered','Weight_in_gms'],
}
FS['FS2'] = [f for f in FS['FS1'] if f not in ('Customer_care_calls','Customer_rating')]
FS['FS3'] = [f for f in FS['FS2'] if f not in ('Prior_purchases','Gender')]
assert [len(FS[k]) for k in ('FS1','FS2','FS3')] == [10, 8, 6]
for k,v in FS.items(): print(k, len(v), v)

FS1 10 ['Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms']
FS2 8 ['Warehouse_block', 'Mode_of_Shipment', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms']
FS3 6 ['Warehouse_block', 'Mode_of_Shipment', 'Cost_of_the_Product', 'Product_importance', 'Discount_offered', 'Weight_in_gms']


## 3. Stratified split 80/20 + lưu danh sách ID
Split **một lần** theo ID, dùng chung cho cả 3 FS. Lưu `splits/split_seed_<seed>.csv` để tái lập.

In [4]:
def make_split(seed):
    tr, te = train_test_split(df, test_size=0.20, stratify=df[TARGET], random_state=seed)
    out = pd.concat([pd.DataFrame({'ID': tr.ID, 'set': 'train'}), pd.DataFrame({'ID': te.ID, 'set': 'test'})])
    out.to_csv(f'splits/split_seed_{seed}.csv', index=False)
    return tr.reset_index(drop=True), te.reset_index(drop=True)

SPLITS = {s: make_split(s) for s in SEEDS}
tr, te = SPLITS[MAIN_SEED]
print(len(tr), len(te), 'late rate train/test:', round(tr[TARGET].mean(),4), round(te[TARGET].mean(),4))
assert set(tr.ID).isdisjoint(te.ID)

8799 2200 late rate train/test: 0.5967 0.5968


## 4. Preprocessing (ColumnTransformer + Pipeline)
`scale=True` cho LR (StandardScaler), `scale=False` cho DT/RF. Imputer giữ lại cho đúng protocol dù hiện chưa có missing.

In [5]:
def make_preprocessor(features, scale):
    cat = [f for f in features if f in CAT]; num = [f for f in features if f in NUM]
    num_steps = [('imp', SimpleImputer(strategy='median'))] + ([('sc', StandardScaler())] if scale else [])
    return ColumnTransformer([
        ('num', Pipeline(num_steps), num),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('ohe', OneHotEncoder(handle_unknown='ignore', drop=None))]), cat)])

def make_model(name, seed, **params):
    if name == 'Dummy': return DummyClassifier(strategy='prior')
    if name == 'LR': return LogisticRegression(max_iter=2000, random_state=seed, **params)
    if name == 'DT': return DecisionTreeClassifier(random_state=seed, **params)
    if name == 'RF': return RandomForestClassifier(random_state=seed, n_jobs=-1, **params)

def make_pipeline(name, fs, seed, **params):
    return Pipeline([('prep', make_preprocessor(FS[fs], scale=(name in ('LR','Dummy')))),
                     ('clf', make_model(name, seed, **params))])

# Hyperparameter grids theo đề xuất của thầy
GRIDS = {
 'LR': {'clf__C':[0.01,0.1,1,10], 'clf__class_weight':[None,'balanced']},
 'DT': {'clf__max_depth':[3,5,10,None], 'clf__min_samples_split':[2,5,10], 'clf__min_samples_leaf':[1,5,10]},
 'RF': {'clf__n_estimators':[100,200,300], 'clf__max_depth':[5,10,20,None],
        'clf__min_samples_leaf':[1,5,10], 'clf__class_weight':[None,'balanced']},
}
def cv(seed): return StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

## 5. Evaluation pipeline
Tất cả metric tính theo lớp Late = positive. Gồm FPR, FNR, Brier, Confusion Matrix, ngưỡng và Top-K.

In [6]:
def evaluate(y_true, proba, threshold=0.5):
    y_true = np.asarray(y_true); pred = (np.asarray(proba) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return dict(threshold=threshold, Accuracy=accuracy_score(y_true,pred),
        Precision=precision_score(y_true,pred,zero_division=0), Recall=recall_score(y_true,pred),
        F1=f1_score(y_true,pred), ROC_AUC=roc_auc_score(y_true,proba), AP=average_precision_score(y_true,proba),
        FPR=fp/(fp+tn), FNR=fn/(fn+tp), Brier=brier_score_loss(y_true,proba),
        TP=tp, TN=tn, FP=fp, FN=fn, AlertRate=pred.mean())

THRESHOLDS = [0.2,0.3,0.4,0.5,0.6,0.7,0.8]
def threshold_table(y_true, proba):
    return pd.DataFrame([evaluate(y_true, proba, t) for t in THRESHOLDS])[
        ['threshold','Precision','Recall','F1','FPR','AlertRate','TP','FP','FN','TN']]

def topk_table(y_true, proba, ks=(0.05,0.10,0.20,0.30,0.50)):
    y = np.asarray(y_true); order = np.argsort(-np.asarray(proba), kind='stable'); base = y.mean(); rows=[]
    for k in ks:
        n = int(round(k*len(y))); hit = y[order[:n]].sum()
        rows.append(dict(TopK=k, n_flagged=n, Precision_at_K=hit/n, Recall_at_K=hit/y.sum(),
                         Lift_at_K=(hit/n)/base, Late_identified=int(hit)))
    return pd.DataFrame(rows)

## 6. Smoke test (CHỈ CV trên Train, không chạm Test)
Mục đích: kiểm tra pipeline chạy đúng, không leak, ra số hợp lý. **Đây không phải kết quả EXP01.**

In [7]:
X_tr, y_tr = tr, tr[TARGET]
rows = []
for name, params in [('Dummy',{}), ('LR',{'C':1}), ('DT',{'max_depth':5}), ('RF',{'n_estimators':100,'max_depth':10})]:
    for fs in ('FS1','FS2','FS3'):
        pipe = make_pipeline(name, fs, MAIN_SEED, **params)
        oof = cross_val_predict(pipe, X_tr[FS[fs]], y_tr, cv=cv(MAIN_SEED), method='predict_proba')[:,1]
        m = evaluate(y_tr, oof); rows.append(dict(model=name, fs=fs, **{k:m[k] for k in ('Precision','Recall','F1','ROC_AUC','AP')}))
smoke = pd.DataFrame(rows).round(4); smoke

,model,fs,Precision,Recall,F1,ROC_AUC,AP
0,Dummy,FS1,0.5967,1.0000,0.7474,0.4999,0.5966
1,Dummy,FS2,0.5967,1.0000,0.7474,0.4999,0.5966
2,Dummy,FS3,0.5967,1.0000,0.7474,0.4999,0.5966
3,LR,FS1,0.7043,0.6792,0.6916,0.7202,0.8371
4,LR,FS2,0.7018,0.6777,0.6895,0.7185,0.8357
5,LR,FS3,0.6988,0.6796,0.6891,0.7158,0.8340
6,DT,FS1,0.9347,0.5046,0.6554,0.7403,0.8474
7,DT,FS2,0.9256,0.5122,0.6595,0.7448,0.8488
8,DT,FS3,0.9400,0.5011,0.6537,0.7425,0.8486
9,RF,FS1,0.8282,0.5556,0.6651,0.7346,0.8502


## 7. Kiểm tra leakage của pipeline (test tự động)

In [8]:
# (a) train/test không trùng ID, (b) mọi split dùng đúng 8799/2200, (c) preprocessor fit chỉ trên train fold
for s,(a,b) in SPLITS.items():
    assert len(a)==8799 and len(b)==2200 and set(a.ID).isdisjoint(b.ID), s
p = make_pipeline('LR','FS2',MAIN_SEED, C=1).fit(tr[FS['FS2']], tr[TARGET])
sc = p.named_steps['prep'].named_transformers_['num'].named_steps['sc']
assert np.allclose(sc.mean_, tr[[f for f in FS['FS2'] if f in NUM]].mean().values)   # scaler học từ train, không từ test
print('OK: split sizes, disjoint IDs, scaler fitted on train only')

OK: split sizes, disjoint IDs, scaler fitted on train only


In [9]:
# Lưu cấu hình thực nghiệm để tái lập
cfg = dict(seeds=SEEDS, main_seed=MAIN_SEED, python=sys.version.split()[0], sklearn=sklearn.__version__,
           pandas=pd.__version__, numpy=np.__version__, feature_sets=FS, grids={k:{a:[str(x) for x in b] for a,b in v.items()} for k,v in GRIDS.items()},
           cv='StratifiedKFold(5, shuffle=True)', thresholds=THRESHOLDS)
json.dump(cfg, open('results/config.json','w'), indent=2, ensure_ascii=False); smoke.to_csv('results/smoke_test_cv_train_only.csv', index=False)